# Interoperabilidad con Qiskit y OpenQASM

Polypus no exige abandonar herramientas ya conocidas: acepta circuitos de Qiskit directamente, y puede exportar e importar OpenQASM, el formato de texto estándar para circuitos cuánticos, tanto en la versión 2.0 como en la 3.

## Ejecutar un circuito de Qiskit directamente

`run_quantum_circuit` acepta un `QuantumCircuit` de Qiskit tal cual, sin conversión previa:

In [ ]:
import polypus
from qiskit import QuantumCircuit

qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)
qc_qiskit.cx(0, 1)
qc_qiskit.measure_all()

result = polypus.run_quantum_circuit(qc_qiskit, shots=1000, infrastructure="local")
print(result.counts[0])

El resultado es el mismo estado de Bell de [`01_primer_circuito.ipynb`](01_primer_circuito.ipynb), `'00'` y `'11'` únicamente, construido esta vez con la API de Qiskit en lugar de con `polypus.Circuit`.

## Exportar a OpenQASM

Cualquier `Circuit` de Polypus se puede exportar a OpenQASM 2.0, el mismo formato de texto que usa Qiskit:

In [ ]:
bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
print(bell.to_qasm2())

El resultado es texto plano, legible y portable a cualquier herramienta que entienda OpenQASM 2.0.

## Importar desde OpenQASM

`Circuit.from_qasm2` hace el camino inverso, y además garantiza una ida y vuelta exacta: exportar, importar y volver a exportar produce el mismo texto, byte a byte.

In [ ]:
qasm_text = bell.to_qasm2()
bell_reimportado = polypus.Circuit.from_qasm2(qasm_text)

print(bell_reimportado.to_qasm2() == qasm_text)

Un circuito importado es un `Circuit` normal: se le pueden seguir encadenando puertas como a cualquier otro.

## OpenQASM 3

Polypus también exporta e importa OpenQASM 3, la versión más reciente del formato, con `to_qasm3` y `Circuit.from_qasm3`. La ida y vuelta también es exacta: exportar, importar y volver a exportar produce el mismo texto, byte a byte.

In [ ]:
texto_qasm3 = bell.to_qasm3()
print(texto_qasm3)

bell_desde_qasm3 = polypus.Circuit.from_qasm3(texto_qasm3)
print(bell_desde_qasm3.to_qasm3() == texto_qasm3)

La diferencia principal con OpenQASM 2.0 es que la versión 3 también puede guardar parámetros libres, ángulos que se dejan abiertos para darles valor más adelante, que se presentan en [`06a_entrenamiento_variacional.ipynb`](06a_entrenamiento_variacional.ipynb). Polypus admite la parte de OpenQASM 3 que describe un circuito como una secuencia lineal de puertas con las medidas al final: un programa con condicionales o bucles, por ejemplo, se rechaza con un `ValueError` que indica la construcción y su línea.

## Traer un circuito de Qiskit por QASM

`from_qasm2` también acepta el texto que produce Qiskit con `qasm2.dumps`, no solo el propio de Polypus. Es útil para convertir un circuito de Qiskit en un `Circuit` nativo, por ejemplo para seguir construyéndolo con el encadenado de métodos visto en [`00a`](00a_programacion_basica.ipynb):

In [ ]:
from qiskit import qasm2

qasm_de_qiskit = qasm2.dumps(qc_qiskit)
circuito_nativo = polypus.Circuit.from_qasm2(qasm_de_qiskit)

print(circuito_nativo.to_qasm2())

Al importar, Polypus mantiene cada puerta bajo su propio nombre, `p`, `u1`, `u2`, `u` e `id` incluidas: solo los builtins `U`/`CX` del propio QASM se normalizan a `u`/`cx`. El resultado sigue siendo el mismo circuito.

Además de OpenQASM, Polypus puede exportar a QIR, un formato de más bajo nivel pensado para compiladores. Queda fuera del alcance de esta serie; se explica en la [guía de circuitos](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/circuits.md#export) (en inglés).

## Dibujar con Qiskit

[`02a`](02a_puertas_y_construccion.ipynb) mostró que un circuito de Polypus se dibuja solo. Quien prefiera los dibujos de matplotlib de Qiskit puede seguir pasando por Qiskit: exportar a QASM, importar de vuelta con `qasm2.loads`, la función inversa de `qasm2.dumps` ya usada arriba, y llamar a `draw("mpl")`, que necesita tener instalados `matplotlib` y `pylatexenc`. El parser de Qiskit no reconoce algunas de las puertas que exporta Polypus, como `swap`, `cp`, `rzz` o `rxx`, salvo que se le pase `custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS`:

```python
circuito_qiskit = qasm2.loads(
    bell.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
circuito_qiskit.draw("mpl")
```

## Resumen

Este notebook ha mostrado varios caminos de interoperabilidad: ejecutar un circuito de Qiskit directamente, exportar un circuito de Polypus a OpenQASM 2.0 y a OpenQASM 3, importar tanto el OpenQASM propio como el que genera Qiskit, y dibujar un circuito de Polypus con matplotlib a través de Qiskit, como alternativa al dibujo nativo. Quien ya sepa Qiskit y quiera el mapeo completo, puertas, ejecución y entrenamiento incluidos, tiene la guía de referencia [`de_qiskit_a_polypus.ipynb`](de_qiskit_a_polypus.ipynb).

La siguiente sección trata sobre dónde se ejecutan estos circuitos: en local, o repartidos entre varios simuladores o QPUs.

## Siguiente paso

Continuar con: [`03a_ejecucion_local.ipynb`](03a_ejecucion_local.ipynb).